<h1 align="center">Practice 6. Data Transformation & EDA: Pandas GroupBy, joins, reshaping (pivot/melt) +
Matplotlib/Seaborn plots.</h1>

### Imports 

In [55]:
import pandas as pd
from sqlalchemy import create_engine, inspect, select, func
from sqlalchemy.ext.automap import automap_base


## Task 1. Database connection

In [9]:
engine = create_engine("sqlite:///ecommerce.sqlite")

In [10]:
inspector = inspect(engine)

In [11]:
inspector.get_table_names()

['customers',
 'order_items',
 'orders',
 'payments',
 'products',
 'returns',
 'shipments']

In [18]:
Base = automap_base()
Base.prepare(autoload_with=engine)

In [19]:
Base.classes.keys()

['returns',
 'customers',
 'order_items',
 'products',
 'payments',
 'shipments',
 'orders']

In [22]:
Customer = Base.classes.customers

In [63]:
Order = Base.classes.orders

In [81]:
Payment = Base.classes.payments

In [27]:
inspector.get_columns("customers")

[{'name': 'customer_id',
  'type': INTEGER(),
  'nullable': True,
  'default': None,
  'primary_key': 1},
 {'name': 'first_name',
  'type': TEXT(),
  'nullable': False,
  'default': None,
  'primary_key': 0},
 {'name': 'last_name',
  'type': TEXT(),
  'nullable': False,
  'default': None,
  'primary_key': 0},
 {'name': 'email',
  'type': TEXT(),
  'nullable': False,
  'default': None,
  'primary_key': 0},
 {'name': 'phone',
  'type': TEXT(),
  'nullable': True,
  'default': None,
  'primary_key': 0},
 {'name': 'country',
  'type': TEXT(),
  'nullable': False,
  'default': None,
  'primary_key': 0},
 {'name': 'city',
  'type': TEXT(),
  'nullable': False,
  'default': None,
  'primary_key': 0},
 {'name': 'signup_date',
  'type': TEXT(),
  'nullable': False,
  'default': None,
  'primary_key': 0},
 {'name': 'loyalty_tier',
  'type': TEXT(),
  'nullable': False,
  'default': None,
  'primary_key': 0},
 {'name': 'marketing_opt_in',
  'type': INTEGER(),
  'nullable': False,
  'default': Non

In [13]:
customers_df = pd.read_sql("customers", engine)

In [14]:
customers_df.head()

,customer_id,first_name,last_name,email,phone,country,city,signup_date,loyalty_tier,marketing_opt_in
0,1,Jill,Ellis,jill.ellis1@example.com,NaN,France,Marseille,2022-08-06,silver,0
1,2,Darrell,Gilbert,darrell.gilbert2@example.com,+1-873-555-0199,United States,New York,2023-01-02,none,0
2,3,Phillip,Francis,phillip.francis3@example.com,+44 7700 900107,United Kingdom,Leeds,2023-05-05,none,1
3,4,Heather,Reyes,heather.reyes4@example.com,NaN,France,Paris,2022-05-27,none,0
4,5,Brent,Mayer,brent.mayer5@example.com,+1-514-555-0123,United States,Houston,2023-10-31,none,1


In [15]:
customers_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26000 entries, 0 to 25999
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   customer_id       26000 non-null  int64
 1   first_name        26000 non-null  str  
 2   last_name         26000 non-null  str  
 3   email             26000 non-null  str  
 4   phone             17355 non-null  str  
 5   country           26000 non-null  str  
 6   city              26000 non-null  str  
 7   signup_date       26000 non-null  str  
 8   loyalty_tier      26000 non-null  str  
 9   marketing_opt_in  26000 non-null  int64
dtypes: int64(2), str(8)
memory usage: 4.1 MB


In [16]:
customers_df.isna().sum()

customer_id            0
first_name             0
last_name              0
email                  0
phone               8645
country                0
city                   0
signup_date            0
loyalty_tier           0
marketing_opt_in       0
dtype: int64

## Task 2. SQL queries


In [40]:
def select_customers_with_phone():
    customers = select(Customer.customer_id, Customer.phone).where(Customer.phone != None)
    return customers

In [41]:
customers_with_phone = pd.read_sql(select_customers_with_phone(), engine)

In [42]:
customers_with_phone.head()

,customer_id,phone
0,2,+1-873-555-0199
1,3,+44 7700 900107
2,5,+1-514-555-0123
3,6,+1-892-555-0159
4,7,+1-331-555-0112


In [44]:
def select_customers_recently_signed():
    customers = select(
        Customer.customer_id, 
        Customer.email, 
        Customer.signup_date
    ).order_by(Customer.signup_date.desc())
    return customers

In [46]:
customers_recently_signed = pd.read_sql(select_customers_recently_signed(), engine)

In [53]:
customers_recently_signed["signup_date"].value_counts().sort_index(ascending=False)

signup_date
2023-12-30    38
2023-12-29    26
2023-12-28    37
2023-12-27    29
2023-12-26    28
              ..
2021-06-05    26
2021-06-04    34
2021-06-03    31
2021-06-02    25
2021-06-01    30
Name: count, Length: 943, dtype: int64

In [54]:
customers_recently_signed.head()

,customer_id,email,signup_date
0,866,richard.cooke866@example.com,2023-12-30
1,2062,tanner.key2062@example.com,2023-12-30
2,2262,jason.kidd2262@example.com,2023-12-30
3,2953,laura.key2953@example.com,2023-12-30
4,3071,michelle.duarte3071@example.com,2023-12-30


In [60]:
def select_customers_by_country():
    customers = select(
        Customer.country,
        func.count().label("customer_count")
    ).group_by(
        Customer.country
    ).order_by(
        func.count().desc()
    )
    return customers

In [65]:
customers_by_country_df = pd.read_sql(select_customers_by_country(), engine)

In [66]:
customers_by_country_df.head()

,country,customer_count
0,United States,10021
1,United Kingdom,3104
2,Germany,2302
3,Canada,2263
4,France,1850


In [78]:
def select_customers_order_count():
    customers = select(
        Customer.customer_id,
        Customer.first_name,
        Customer.city,
        func.count(Order.order_id).label("orders_count")
    ).join(
        Order,
        Order.customer_id == Customer.customer_id
    ).group_by(
        Customer.customer_id
    ).order_by(
        func.count(Order.order_id).desc()
    )
    return customers

In [79]:
customers_order_count_df = pd.read_sql(select_customers_order_count(), engine)

In [80]:
customers_order_count_df.head()

,customer_id,first_name,city,orders_count
0,15091,Barbara,Toronto,126
1,24489,Monique,Barcelona,87
2,20141,Anne,Toronto,74
3,5567,Jeanette,Marseille,71
4,4268,Brandi,Montreal,64


## Task 3. Data transformation

In [109]:
customer_payment_query = select(
    Customer.customer_id,
    Customer.first_name,
    Customer.last_name,
    Customer.email,
    Customer.signup_date,
    Customer.marketing_opt_in,
    func.sum(Payment.amount).label("total_spend")
).join(
    Order,
    Order.customer_id == Customer.customer_id
).join(
    Payment,
    Payment.order_id == Order.order_id
).where(
    Payment.status == "captured"
).group_by(
    Customer.customer_id
)

In [110]:
customer_payment_df = pd.read_sql(customer_payment_query, engine)

In [111]:
customer_payment_df

,customer_id,first_name,last_name,email,signup_date,marketing_opt_in,total_spend
0,1,Jill,Ellis,jill.ellis1@example.com,2022-08-06,0,1475.33
1,2,Darrell,Gilbert,darrell.gilbert2@example.com,2023-01-02,0,4307.32
2,3,Phillip,Francis,phillip.francis3@example.com,2023-05-05,1,5025.89
3,4,Heather,Reyes,heather.reyes4@example.com,2022-05-27,0,4129.99
4,5,Brent,Mayer,brent.mayer5@example.com,2023-10-31,1,3768.76
...,...,...,...,...,...,...,...
25319,25996,Jessica,Beasley,jessica.beasley25996@example.com,2023-09-16,1,3162.66
25320,25997,Crystal,Hoover,crystal.hoover25997@example.com,2023-07-14,0,218.18
25321,25998,Tom,Graham,tom.graham25998@example.com,2022-07-16,1,2503.12
25322,25999,Gene,Duran,gene.duran25999@example.com,2023-05-22,1,674.69


In [112]:
customer_payment_df.isna().sum()

customer_id         0
first_name          0
last_name           0
email               0
signup_date         0
marketing_opt_in    0
total_spend         0
dtype: int64

In [117]:
customer_payment_df.dtypes

customer_id                  int64
first_name                     str
last_name                      str
email                          str
signup_date         datetime64[us]
marketing_opt_in             int64
total_spend                float64
fullname                       str
dtype: object

In [116]:
customer_payment_df["signup_date"] = pd.to_datetime(customer_payment_df["signup_date"])

In [119]:
customer_payment_df.dtypes

customer_id                  int64
first_name                     str
last_name                      str
email                          str
signup_date         datetime64[us]
marketing_opt_in             int64
total_spend                float64
fullname                       str
dtype: object

In [120]:
customer_payment_df["fullname"] = customer_payment_df["first_name"] + " " + customer_payment_df["last_name"]

In [121]:
customer_payment_df.head()

,customer_id,first_name,last_name,email,signup_date,marketing_opt_in,total_spend,fullname
0,1,Jill,Ellis,jill.ellis1@example.com,2022-08-06,0,1475.33,Jill Ellis
1,2,Darrell,Gilbert,darrell.gilbert2@example.com,2023-01-02,0,4307.32,Darrell Gilbert
2,3,Phillip,Francis,phillip.francis3@example.com,2023-05-05,1,5025.89,Phillip Francis
3,4,Heather,Reyes,heather.reyes4@example.com,2022-05-27,0,4129.99,Heather Reyes
4,5,Brent,Mayer,brent.mayer5@example.com,2023-10-31,1,3768.76,Brent Mayer


In [123]:
customer_payment_df.drop(columns=["first_name", "last_name"], inplace=True)

In [125]:
customer_payment_df.head()

,customer_id,email,signup_date,marketing_opt_in,total_spend,fullname
0,1,jill.ellis1@example.com,2022-08-06,0,1475.33,Jill Ellis
1,2,darrell.gilbert2@example.com,2023-01-02,0,4307.32,Darrell Gilbert
2,3,phillip.francis3@example.com,2023-05-05,1,5025.89,Phillip Francis
3,4,heather.reyes4@example.com,2022-05-27,0,4129.99,Heather Reyes
4,5,brent.mayer5@example.com,2023-10-31,1,3768.76,Brent Mayer


## Task 4. ETL pipeline

In [6]:
%run etl_pipeline.py

   product_id        sku                     product_name       category  \
0           1  OUT-00001           Nomad Down Vest (Sand)        Apparel   
1           2  OUT-00002  Ridge Family Dome Tent (Cobalt)          Tents   
2           3  OUT-00003   Ridge Ultralight Tent (Cobalt)          Tents   
3           4  OUT-00004     Basecamp Sleeping Pad (Moss)  Sleeping Bags   
4           5  OUT-00005     Tundra Camp Lantern (Cobalt)       Lighting   

            brand  unit_price  unit_cost  weight_kg  is_active created_at  \
0      Stonebrook      161.96      79.21       4.68       True 2022-04-18   
1  Ironwood Trail      370.93     236.28       3.45       True 2022-02-09   
2        Larkspur      145.98     101.21       4.91      False 2021-07-31   
3          Halden      230.20     132.45       3.94       True 2023-03-27   
4       Wildhaven       16.81       7.93       4.14       True 2022-04-14   

    color  
0    Sand  
1  Cobalt  
2  Cobalt  
3    Moss  
4  Cobalt  


## Task 5. Data Check

In [127]:
products_clean = pd.read_sql("products_clean", engine)

In [128]:
products_raw = pd.read_csv("./products.csv")

In [129]:
len(products_clean)

1500

In [130]:
len(products_raw)

1500

In [131]:
products_clean.isna().sum()

product_id       0
sku              0
product_name     0
category         0
brand            0
unit_price       0
unit_cost        0
weight_kg       78
is_active        0
created_at       0
color            0
dtype: int64

In [132]:
products_raw.isna().sum()

product_id       0
sku              0
product_name     0
category         0
brand            0
unit_price       0
unit_cost        0
weight_kg       78
is_active        0
created_at       0
dtype: int64

In [134]:
products_clean.duplicated().sum()

np.int64(0)

In [135]:
products_raw.duplicated().sum()

np.int64(0)

In [136]:
products_clean.dtypes

product_id               int64
sku                        str
product_name               str
category                   str
brand                      str
unit_price             float64
unit_cost              float64
weight_kg              float64
is_active                 bool
created_at      datetime64[us]
color                      str
dtype: object

In [137]:
products_raw.dtypes

product_id        int64
sku                 str
product_name        str
category            str
brand               str
unit_price      float64
unit_cost       float64
weight_kg       float64
is_active         int64
created_at          str
dtype: object

### Findings 

1. **Data types:** Initial raw data from csv had incorrect data types, while reading from sql tables preserves its dtypes
2. **Missing values:** Both data sets had missing values that are more likely to be keeped. Though if it wasn't the case ETL process could handle them in defferent way.
3. **Columns:** With ETL pipeline I normalized product_name column to keep only product name itself, while seperating additional information about it's color into different column color.
4. **Storage:** Storing data in sql database has much more benefits compared to raw csv files. It occupies less disk space and retrieves data faster.
5. **Record count:** After ETL process no data were abonded.